In [ ]:
import pandas as pd
from pathlib import Path

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]


dates = (
    pd.read_parquet(DATES_PATH, columns=["date"])
      .dropna()
      .sort_values("date")["date"]
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

Nb dates: 791
Date min: 1960-01-01 00:00:00 | Date max: 2025-11-01 00:00:00


In [ ]:
entity_df = (
    pd.MultiIndex.from_product(
        [series_ids, dates],
        names=["series_id", "date"]
    )
    .to_frame(index=False)
)

entity_df.head()

,series_id,date
0,BUSLOANS,1960-01-01
1,BUSLOANS,1960-02-01
2,BUSLOANS,1960-03-01
3,BUSLOANS,1960-04-01
4,BUSLOANS,1960-05-01


In [3]:
from pathlib import Path
from feast import FeatureStore

REPO_PATH = (
    Path.cwd()
    .parent
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

store = FeatureStore(repo_path=str(REPO_PATH))

print("Project:", store.project)
print("Feature views:", [fv.name for fv in store.list_feature_views()])

Project: unemployment_feature_store
Feature views: ['stationary_value', 'raw_value']


In [4]:
df_stationary = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "stationary_value:value",
    ],
).to_df()

df_stationary.head()

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


,series_id,date,value
0,BUSLOANS,1960-01-01 00:00:00+00:00,0.011578
1,USREC,1960-01-01 00:00:00+00:00,0.000000
2,M2SL,1960-01-01 00:00:00+00:00,0.001323
3,DPCERA3M086SBEA,1960-01-01 00:00:00+00:00,0.001204
4,UNRATE,1960-01-01 00:00:00+00:00,-0.800000


In [5]:
# --------------------------------------------------
# Pivot LONG → WIDE (1 colonne par série)
# --------------------------------------------------
df_wide = (
    df_stationary
    .rename(columns={"stationary_value__value": "value"})
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

df_wide.head()

series_id,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,UNRATE,USREC
date,,,,,,,,,,,
1960-01-01 00:00:00+00:00,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,-0.8,0.0
1960-02-01 00:00:00+00:00,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,-1.1,0.0
1960-03-01 00:00:00+00:00,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,-0.2,0.0
1960-04-01 00:00:00+00:00,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0,0.0
1960-05-01 00:00:00+00:00,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,0.0,1.0
